In [ ]:
import os
import zipfile
import duckdb
import pandas as pd
from google.colab import drive

#functions
def extract_zip_to_ssd(zip_name, source_dir, target_dir):
    zip_path = os.path.join(source_dir, zip_name)
    print(f"extracting {zip_name} to local ssd disk")
    with zipfile.ZipFile(zip_path, 'r') as z:
        extracted_fname = z.namelist()[0]
        z.extractall(target_dir)
    extracted_path = os.path.join(target_dir, extracted_fname)
    print(f"saved uncompressed: {extracted_fname}")
    return extracted_path

def print_table_count(con, table_name, label):
    count = con.execute(f"SELECT COUNT(*) FROM {table_name};").fetchone()[0]
    print(f"[diagnostic] {label} ({table_name}): {count:,} rows")

#paths setup
print("ssd extraction")

drive_mount_point = os.path.join(os.sep, "content", "drive")
drive.mount(drive_mount_point, force_remount=False)

drive_data_dir = os.path.join(drive_mount_point, "MyDrive", "data")
ssd_dir = os.path.join(os.sep, "tmp", "uspto_uncompressed")
os.makedirs(ssd_dir, exist_ok=True)

db_path = os.path.join(ssd_dir, "clean_pipeline.db")
if os.path.exists(db_path):
    os.remove(db_path)

con = duckdb.connect(db_path)
con.execute("SET memory_limit = '4GB';")
con.execute("SET threads = 2;")

#load and sample cohort
dta_path = os.path.join(drive_data_dir, "patentvc_enhanced_4var.dta")
print("loading target cohort")
df_stata = pd.read_stata(dta_path)
con.register("df_stata_temp", df_stata)

con.execute("""
    CREATE OR REPLACE TABLE filtered_2000_2020 AS
    SELECT
        CAST(patent_number AS VARCHAR) AS patent_id,
        CAST(application_date AS DATE) AS application_date,
        CAST(grant_date AS DATE) AS grant_date,
        EXTRACT(YEAR FROM CAST(grant_date AS DATE)) AS grant_year,
        EXTRACT(YEAR FROM CAST(application_date AS DATE)) AS app_year,
        VC AS vc_backed
    FROM df_stata_temp
    WHERE EXTRACT(YEAR FROM CAST(grant_date AS DATE)) BETWEEN 2000 AND 2020;

    CREATE OR REPLACE TABLE target_patents AS
    SELECT * FROM filtered_2000_2020
    USING SAMPLE 500000 (reservoir);

    DROP TABLE filtered_2000_2020;
""")
del df_stata
con.unregister("df_stata_temp")

print_table_count(con, "target_patents", "target cohort sampled")

#process cpc classes
cpc_file = extract_zip_to_ssd("g_cpc_current.tsv.zip", drive_data_dir, ssd_dir)
print("processing cpc classes")
print_table_count(con, "target_patents", "pre-merge base count for cpc")

con.execute(f"""
    CREATE OR REPLACE TABLE target_cpcs AS
    SELECT
        p.patent_id,
        FIRST(CAST(c.cpc_section AS VARCHAR)) AS cpc_section,
        FIRST(CAST(c.cpc_subclass AS VARCHAR)) AS cpc_subclass
    FROM target_patents p
    INNER JOIN read_csv('{cpc_file}', header=True, delim='\t', all_varchar=True, ignore_errors=True) c
       ON p.patent_id = c.patent_id
    GROUP BY p.patent_id;
""")

os.remove(cpc_file)
print_table_count(con, "target_cpcs", "post-merge matched cpc records")
print("cpc classes matched")

#process citations
cite_file = extract_zip_to_ssd("g_us_patent_citation.tsv.zip", drive_data_dir, ssd_dir)
print("processing citations on local ssd")
print_table_count(con, "target_patents", "pre-merge base count for citations")

con.execute(f"""
    CREATE OR REPLACE TABLE raw_citations AS
    SELECT
        CAST(patent_id AS VARCHAR) AS focal_id,
        CAST(citation_patent_id AS VARCHAR) AS cited_id
    FROM read_csv('{cite_file}', header=True, delim='\t', all_varchar=True, ignore_errors=True);

    CREATE OR REPLACE TABLE target_backward_cites AS
    SELECT p.patent_id, COUNT(DISTINCT c.cited_id) AS backward_pat_citations
    FROM target_patents p
    INNER JOIN raw_citations c ON p.patent_id = c.focal_id
    GROUP BY p.patent_id;

    CREATE OR REPLACE TABLE target_forward_cites_raw AS
    SELECT p.patent_id, COUNT(DISTINCT c.focal_id) AS raw_forward_citations
    FROM target_patents p
    INNER JOIN raw_citations c ON p.patent_id = c.cited_id
    GROUP BY p.patent_id;

    DROP TABLE raw_citations;
""")

os.remove(cite_file)
print_table_count(con, "target_backward_cites", "post-merge matched backward citations")
print_table_count(con, "target_forward_cites_raw", "post-merge matched forward citations")
print("citation networks matched")

#process npl
npl_file = extract_zip_to_ssd("g_other_reference.tsv.zip", drive_data_dir, ssd_dir)
print("processing non-patent literature")
print_table_count(con, "target_patents", "pre-merge base count for npl")

con.execute(f"""
    CREATE OR REPLACE TABLE target_npl_cites AS
    SELECT
        p.patent_id,
        COUNT(DISTINCT n.other_reference_text) AS npl_citations
    FROM target_patents p
    INNER JOIN read_csv('{npl_file}', header=True, delim='\t', all_varchar=True, ignore_errors=True) n
       ON p.patent_id = n.patent_id
    GROUP BY p.patent_id;
""")

os.remove(npl_file)
print_table_count(con, "target_npl_cites", "post-merge matched npl citations")
print("non-patent literature matched")

#process assignees
assignee_file = extract_zip_to_ssd("g_assignee_disambiguated.tsv.zip", drive_data_dir, ssd_dir)
print("processing assignees")
print_table_count(con, "target_patents", "pre-merge base count for assignees")

con.execute(f"""
    CREATE OR REPLACE TABLE target_assignees AS
    SELECT
        p.patent_id,
        FIRST(CAST(a.disambig_assignee_organization AS VARCHAR)) AS assignee_organization,
        FIRST(CAST(a.assignee_type AS VARCHAR)) AS assignee_type
    FROM target_patents p
    INNER JOIN read_csv('{assignee_file}', header=True, delim='\t', all_varchar=True, ignore_errors=True) a
       ON p.patent_id = a.patent_id
    GROUP BY p.patent_id;
""")

os.remove(assignee_file)
print_table_count(con, "target_assignees", "post-merge matched assignees")
print("assignees matched")

#master assembly
print("assembling final step1_metrics table")
print_table_count(con, "target_patents", "pre-assembly base table count")

con.execute("""
    CREATE OR REPLACE TABLE step1_metrics AS
    SELECT
        p.*,
        COALESCE(cpc.cpc_section, 'UNKNOWN') AS cpc_section,
        COALESCE(cpc.cpc_subclass, 'UNKNOWN') AS cpc_subclass,
        COALESCE(b.backward_pat_citations, 0) AS backward_pat_citations,
        COALESCE(f.raw_forward_citations, 0) AS raw_forward_citations,
        COALESCE(n.npl_citations, 0) AS npl_citations,
        a.assignee_organization,
        a.assignee_type
    FROM target_patents p
    LEFT JOIN target_cpcs cpc ON p.patent_id = cpc.patent_id
    LEFT JOIN target_backward_cites b ON p.patent_id = b.patent_id
    LEFT JOIN target_forward_cites_raw f ON p.patent_id = f.patent_id
    LEFT JOIN target_npl_cites n ON p.patent_id = n.patent_id
    LEFT JOIN target_assignees a ON p.patent_id = a.patent_id;
""")

print_table_count(con, "step1_metrics", "post-assembly master table count")

step1_parquet = os.path.join(drive_data_dir, "step1_metrics.parquet")
con.execute(f"COPY step1_metrics TO '{step1_parquet}' (FORMAT PARQUET);")
print(f"step 1 complete. saved to drive: {step1_parquet}")

direct ssd extraction method
Mounted at /content/drive
loading target cohort


KeyboardInterrupt: 